In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Added on_bad_lines='skip' to bypass rows with unexpected number of columns
df = pd.read_csv("house_prices.csv", on_bad_lines='skip')
print(df.head())
print(df.shape)

   Index                                              Title  \
0      0  1 BHK Ready to Occupy Flat for sale in Srushti...   
1      1  2 BHK Ready to Occupy Flat for sale in Dosti V...   
2      2  2 BHK Ready to Occupy Flat for sale in Sunrise...   
3      3        1 BHK Ready to Occupy Flat for sale Kasheli   
4      4  2 BHK Ready to Occupy Flat for sale in TenX Ha...   

                                         Description Amount(in rupees)  \
0  Bhiwandi, Thane has an attractive 1 BHK Flat f...           42 Lac    
1  One can find this stunning 2 BHK flat for sale...           98 Lac    
2  Up for immediate sale is a 2 BHK apartment in ...          1.40 Cr    
3  This beautiful 1 BHK Flat is available for sal...           25 Lac    
4  This lovely 2 BHK Flat in Pokhran Road, Thane ...          1.60 Cr    

   Price (in rupees) location Carpet Area         Status         Floor  \
0             6000.0    thane    500 sqft  Ready to Move  10 out of 11   
1            13799.0    than

/tmp/ipykernel_637/1392675384.py:5: DtypeWarning: Columns (13,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("house_prices.csv", on_bad_lines='skip')


In [3]:
print("Before handling missing values\n",df.isnull().sum())
df = df.drop(columns=[
    "Index",
    "Description",
    "Price (in rupees)",
    "facing",
    "overlooking",
    "Society",
    "Car Parking",
    "Super Area",
    "Dimensions",
    "Plot Area"
])
df=df.drop_duplicates()
print(df.shape)

Before handling missing values
 Index                    0
Title                    0
Description           1105
Amount(in rupees)        1
Price (in rupees)     1897
location                 1
Carpet Area          24736
Status                  57
Floor                 1529
Transaction             16
Furnishing            1019
facing               19982
overlooking          23671
Society              32854
Bathroom                61
Balcony              16383
Car Parking          26317
Ownership            19489
Super Area           17736
Dimensions           41755
Plot Area            41755
dtype: int64
(10779, 11)


In [4]:
# Convert house prices into rupees
def clean_price(price):
    price = str(price).strip().lower()

    if "call for price" in price:
        return np.nan

    if "cr" in price:
        return float(price.replace("cr", "").strip()) * 10000000

    if "lac" in price:
        return float(price.replace("lac", "").strip()) * 100000

    return pd.to_numeric(price, errors="coerce")


df["House_Price"] = df["Amount(in rupees)"].apply(clean_price)
# Remove the original price column
df = df.drop(columns=["Amount(in rupees)"])
# Remove rows where the price is missing
df = df.dropna(subset=["House_Price"])
# Remove invalid prices
df = df[df["House_Price"] > 0].copy()
print(df[["House_Price"]].head())
print("Remaining rows:", len(df))

   House_Price
0    4200000.0
1    9800000.0
2   14000000.0
3    2500000.0
4   16000000.0
Remaining rows: 10402


In [5]:
# Extract bedrooms, such as 2 from "2 BHK"
df["BHK"] = pd.to_numeric(
    df["Title"].str.extract(r"(\d+)\s*BHK")[0],
    errors="coerce"
)
# Extract carpet area, such as 500 from "500 sqft"
df["Carpet_Area"] = pd.to_numeric(
    df["Carpet Area"].astype(str).str.extract(r"([\d,]+\.?\d*)")[0]
    .str.replace(",", "", regex=False),
    errors="coerce"
)
# Convert bathroom and balcony counts into numbers
df["Bathroom"] = pd.to_numeric(df["Bathroom"], errors="coerce")
df["Balcony"] = pd.to_numeric(df["Balcony"], errors="coerce")
# Remove the original columns we no longer need
df = df.drop(columns=[
    "Title", "Carpet Area", "Super Area"
], errors="ignore")
# Remove zero or negative values from property measurements
for col in ["BHK", "Carpet_Area", "Bathroom", "Balcony"]:
    df.loc[df[col] <= 0, col] = np.nan

print(df.head())

  location         Status         Floor Transaction      Furnishing  Bathroom  \
0    thane  Ready to Move  10 out of 11      Resale     Unfurnished       1.0   
1    thane  Ready to Move   3 out of 22      Resale  Semi-Furnished       2.0   
2    thane  Ready to Move  10 out of 29      Resale     Unfurnished       2.0   
3    thane  Ready to Move    1 out of 3      Resale     Unfurnished       1.0   
4    thane  Ready to Move  20 out of 42      Resale     Unfurnished       2.0   

   Balcony             Ownership  House_Price  BHK  Carpet_Area  
0      2.0                   NaN    4200000.0  1.0        500.0  
1      NaN              Freehold    9800000.0  2.0        473.0  
2      NaN              Freehold   14000000.0  2.0        779.0  
3      1.0                   NaN    2500000.0  1.0        530.0  
4      NaN  Co-operative Society   16000000.0  2.0        635.0  


In [6]:
# Fill missing numerical values
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values remaining:", df.isnull().sum().sum())

Missing values remaining: 0


In [12]:
X = df.drop(columns=["House_Price"])
y = df["House_Price"]

X = pd.get_dummies(X, drop_first=True, dtype=int)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:

import joblib

joblib.dump(tree_model, "house_price_model.pkl")
joblib.dump(X_train.columns.tolist(), "model_columns.pkl")

print("Model and columns saved successfully!")


Model and columns saved successfully!


In [15]:

print("Number of input columns:", len(X_train.columns))
print("\nInput column names:")
print(X_train.columns.tolist())


Number of input columns: 739

Input column names:
['Bathroom', 'Balcony', 'BHK', 'Carpet_Area', 'location_bangalore', 'location_chennai', 'location_mumbai', 'location_nagpur', 'location_navi-mumbai', 'location_thane', 'Floor_1 out of 1', 'Floor_1 out of 10', 'Floor_1 out of 11', 'Floor_1 out of 12', 'Floor_1 out of 13', 'Floor_1 out of 14', 'Floor_1 out of 15', 'Floor_1 out of 16', 'Floor_1 out of 17', 'Floor_1 out of 18', 'Floor_1 out of 19', 'Floor_1 out of 2', 'Floor_1 out of 20', 'Floor_1 out of 21', 'Floor_1 out of 22', 'Floor_1 out of 23', 'Floor_1 out of 25', 'Floor_1 out of 26', 'Floor_1 out of 27', 'Floor_1 out of 28', 'Floor_1 out of 29', 'Floor_1 out of 3', 'Floor_1 out of 31', 'Floor_1 out of 33', 'Floor_1 out of 4', 'Floor_1 out of 5', 'Floor_1 out of 6', 'Floor_1 out of 7', 'Floor_1 out of 8', 'Floor_1 out of 88', 'Floor_1 out of 9', 'Floor_10 out of 10', 'Floor_10 out of 11', 'Floor_10 out of 12', 'Floor_10 out of 13', 'Floor_10 out of 14', 'Floor_10 out of 15', 'Floor_1

In [25]:


%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("house_price_model.pkl")
model_columns = joblib.load("model_columns.pkl")

st.title("House Price Prediction 🏠")
st.write("Enter the house details below to estimate its price.")

# House details
bhk = st.number_input("Number of bedrooms (BHK)", 1, 10, 2)
area = st.number_input("Carpet area (sq ft)", 100, 10000, 1000)
bathroom = st.number_input("Number of bathrooms", 1, 10, 2)
balcony = st.number_input("Number of balconies", 0, 5, 1)

# Location dropdown
location_columns = [
    col for col in model_columns
    if col.startswith("location_")
]

locations = ["Other"] + [
    col.replace("location_", "").replace("-", " ").title()
    for col in location_columns
]

selected_location = st.selectbox("Select Location", locations)

if st.button("Predict House Price"):
    input_data = pd.DataFrame(
        0, index=[0], columns=model_columns
    )

    input_data["BHK"] = bhk
    input_data["Carpet_Area"] = area
    input_data["Bathroom"] = bathroom
    input_data["Balcony"] = balcony

    if selected_location != "Other":
        location_column = next(
            col for col in location_columns
            if col.replace("location_", "").replace("-", " ").title()
            == selected_location
        )
        input_data[location_column] = 1

    predicted_price = model.predict(input_data)[0]

    st.success(f"Estimated House Price: ₹{predicted_price:,.0f}")


Overwriting app.py


In [28]:

!pip install -q streamlit pyngrok


In [29]:

import os

print("app.py exists:", os.path.exists("app.py"))
print("Model exists:", os.path.exists("house_price_model.pkl"))
print("Columns file exists:", os.path.exists("model_columns.pkl"))


app.py exists: True
Model exists: True
Columns file exists: True


In [30]:
!pip install -q streamlit pyngrok

In [23]:
from pyngrok import ngrok
from getpass import getpass

token = getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(token)

public_url = ngrok.connect(8501)
print("Your app URL:", public_url)

Paste your ngrok authtoken: ··········
Your app URL: NgrokTunnel: "https://rinsing-payback-remission.ngrok-free.dev" -> "http://localhost:8501"


In [24]:
!streamlit run app.py --server.port 8501 &>/content/streamlit.log &

In [26]:
!pkill -f "streamlit run app.py" || true

^C


In [27]:
!streamlit run app.py --server.port 8501 &>/content/streamlit.log &

In [13]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the model
tree_model = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=20,
    random_state=42
)

# Train the model
tree_model.fit(X_train, y_train)

# Make predictions
tree_predictions = tree_model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, tree_predictions)
rmse = np.sqrt(mean_squared_error(y_test, tree_predictions))
r2 = r2_score(y_test, tree_predictions)

print("Decision Tree Results")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

Decision Tree Results
MAE: 4796545.18
RMSE: 11855298.52
R2 Score: 0.5758
